<a href="https://colab.research.google.com/github/routparam12/Python_qns/blob/main/Yolo-testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q ultralytics deepface

In [2]:
import cv2
from ultralytics import YOLO
from deepface import DeepFace
import numpy as np

# 1. Load a face detection YOLO model (or use a face-specific weights file)
# If you have yolov8n-face.pt, use that; otherwise detect 'person' and crop upper region
detector = YOLO("yolov8n.pt")

# Simple in-memory store: {person_name: embedding_vector}
face_db = {}

def get_face_crop(image_path):
    img = cv2.imread(image_path)
    results = detector.predict(source=img, conf=0.4, verbose=False)
    boxes = results[0].boxes

    if len(boxes) == 0:
        return img  # fallback to full image if no box detected

    # Take highest-confidence box
    best_idx = int(boxes.conf.argmax())
    x1, y1, x2, y2 = map(int, boxes.xyxy[best_idx].tolist())
    return img[y1:y2, x1:x2]

def register_person(name, image_path):
    crop = get_face_crop(image_path)
    emb = DeepFace.represent(
        img_path=crop,
        model_name="Facenet512",
        enforce_detection=False
    )[0]["embedding"]
    face_db[name] = np.array(emb)
    print(f"Stored face for: {name}")

def identify_person(image_path, threshold=0.40):
    crop = get_face_crop(image_path)
    query_emb = np.array(
        DeepFace.represent(
            img_path=crop,
            model_name="Facenet512",
            enforce_detection=False
        )[0]["embedding"]
    )

    best_name = "Unknown"
    best_dist = float("inf")

    for name, ref_emb in face_db.items():
        # Cosine distance (lower = closer match)
        cos_sim = np.dot(query_emb, ref_emb) / (np.linalg.norm(query_emb) * np.linalg.norm(ref_emb))
        dist = 1 - cos_sim
        if dist < best_dist:
            best_dist = dist
            best_name = name

    if best_dist <= threshold:
        print(f"Identified: {best_name} (distance={best_dist:.3f})")
        return best_name
    else:
        print(f"Unknown person (closest={best_name}, distance={best_dist:.3f})")
        return "Unknown"

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


26-11-17 10:23:35 - Directory /root/.deepface has been created
26-11-17 10:23:35 - Directory /root/.deepface/weights has been created


In [3]:
from google.colab import files

# Upload reference photo of Person A
print("Upload reference photo:")
ref_upload = files.upload()
ref_path = next(iter(ref_upload))
register_person("Rohan", ref_path)

# Upload a different photo to identify
print("Upload photo to identify:")
test_upload = files.upload()
test_path = next(iter(test_upload))
identify_person(test_path)

Upload reference photo:


Saving Neem.jpg to Neem.jpg
26-11-17 10:23:48 - 🔗 facenet512_weights.pth will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facenet512_weights.pth to /root/.deepface/weights/facenet512_weights.pth...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facenet512_weights.pth
To: /root/.deepface/weights/facenet512_weights.pth
100%|██████████| 94.3M/94.3M [00:01<00:00, 48.0MB/s]


26-11-17 10:23:51 - 🔗 haarcascade_frontalface_default.xml will be downloaded from https://github.com/opencv/opencv/raw/4.9.0/data/haarcascades/haarcascade_frontalface_default.xml to /root/.deepface/weights/haarcascade_frontalface_default.xml...


Downloading...
From: https://github.com/opencv/opencv/raw/4.9.0/data/haarcascades/haarcascade_frontalface_default.xml
To: /root/.deepface/weights/haarcascade_frontalface_default.xml
930kB [00:00, 66.5MB/s]                   


26-11-17 10:23:51 - 🔗 haarcascade_eye.xml will be downloaded from https://github.com/opencv/opencv/raw/4.9.0/data/haarcascades/haarcascade_eye.xml to /root/.deepface/weights/haarcascade_eye.xml...


Downloading...
From: https://github.com/opencv/opencv/raw/4.9.0/data/haarcascades/haarcascade_eye.xml
To: /root/.deepface/weights/haarcascade_eye.xml
341kB [00:00, 55.6MB/s]                    

Stored face for: Rohan
Upload photo to identify:


Saving Neem.jpg to Neem (1).jpg
Identified: Rohan (distance=0.000)


'Rohan'

In [4]:
!pip install -q transformers torch pillow

In [5]:
import torch
from PIL import Image
from google.colab import files
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

# 1. Upload your image
uploaded = files.upload()
img_path = next(iter(uploaded))
image = Image.open(img_path).convert("RGB")

# 2. Load open-vocabulary detector (GPU recommended in Colab)
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "google/owlv2-base-patch16-ensemble"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

# 3. Pass ANY object names you want to check for
candidate_labels = ["table", "cat", "person", "dog", "chair", "laptop", "phone"]

inputs = processor(text=[candidate_labels], images=image, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)

target_sizes = torch.tensor([image.size[::-1]]).to(device)  # (height, width)
results = processor.post_process_grounded_object_detection(
    outputs=outputs,
    target_sizes=target_sizes,
    threshold=0.20,
    text_labels=[candidate_labels],
)[0]

# 4. Print detected object names
detected_names = set()
for score, label, box in zip(results["scores"], results["text_labels"], results["boxes"]):
    detected_names.add(label)
    coords = [round(x, 1) for x in box.tolist()]
    print(f"Detected: {label} (confidence: {score:.2f}) at {coords}")

if not detected_names:
    print("No matching objects found from candidate_labels.")

Saving Neem.jpg to Neem (2).jpg


preprocessor_config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  620MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/418 [00:00<?, ?it/s]

No matching objects found from candidate_labels.


In [4]:
!pip install --upgrade -q transformers torch torchvision timm

In [6]:
import torch
from PIL import Image
from google.colab import files
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from huggingface_hub import hf_hub_download
import json

# 1. Upload a plant or leaf image
uploaded = files.upload()
img_path = next(iter(uploaded))
image = Image.open(img_path).convert("RGB")

# 2. Load the model directly using native timm
model_id = "PlantNet/mobilenetv4-conv-s-plantclef2024"
model = timm.create_model(f"hf_hub:{model_id}", pretrained=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

# Resolve data preprocessing transforms using timm
config = resolve_data_config({}, model=model)
transform = create_transform(**config)
input_tensor = transform(image).unsqueeze(0).to(device) # [1, 3, H, W]

# Load the configuration file which contains the label names
config_path = hf_hub_download(repo_id=model_id, filename="config.json")
with open(config_path, "r") as f:
    config_json = json.load(f)

label_names = config_json.get("label_names", [])

# Load the local labels mapping file containing true plant taxonomies
labels_path = hf_hub_download(repo_id=model_id, filename="labels.json")
with open(labels_path, "r", encoding="utf-8") as f:
    labels_data = json.load(f)

# Map species_id to botanical information
id_to_info = {item["species_id"]: item for item in labels_data}

def get_plantnet_species_name(plantnet_id):
    """Resolve species name locally using labels.json"""
    species_id = str(plantnet_id)
    if species_id in id_to_info:
        info = id_to_info[species_id]
        species_name = info.get("species", "")
        family_name = info.get("family", "")
        return f"{species_name} [{family_name}]"
    return f"Plant ID: {plantnet_id}"

# 3. Run inference
print("Classifying image...")
with torch.no_grad():
    logits = model(input_tensor)
    probs = torch.softmax(logits, dim=-1)[0]

top_probs, top_indices = torch.topk(probs, k=5)
for rank, (prob, idx) in enumerate(zip(top_probs, top_indices), start=1):
    class_index = idx.item()
    if class_index < len(label_names):
        plant_id = label_names[class_index]
        label = get_plantnet_species_name(plant_id)
    else:
        label = f"Unknown index {class_index}"
    print(f"{rank}. {label} — {prob.item() * 100:.2f}%")

Saving Mango.jpg to Mango (1).jpg


config.json:   0%|          | 0.00/118k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 50.1MB            

model.safetensors: downloading bytes:           |  0.00B            

labels.json:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Classifying image...
1. Aloysia citrodora Paláu [Verbenaceae] — 10.39%
2. Nerium oleander L. [Apocynaceae] — 5.82%
3. Ludwigia grandiflora (Michx.) Greuter & Burdet [Onagraceae] — 3.45%
4. Castanea crenata Siebold & Zucc. [Fagaceae] — 2.56%
5. Schinus terebinthifolia Raddi [Anacardiaceae] — 2.16%


In [34]:
import torch, json, requests
import timm
from PIL import Image
from google.colab import files
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from huggingface_hub import HfApi, hf_hub_download

# 1. Upload image
uploaded = files.upload()
img_path = next(iter(uploaded))
image = Image.open(img_path).convert("RGB")

# 2. Load model
model_id = "PlantNet/mobilenetv4-conv-s-plantclef2024"
model = timm.create_model(f"hf_hub:{model_id}", pretrained=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

cfg = resolve_data_config({}, model=model)
transform = create_transform(**cfg)
input_tensor = transform(image).unsqueeze(0).to(device)

# 3. Labels (GBIF species IDs)
config_path = hf_hub_download(repo_id=model_id, filename="config.json")
with open(config_path) as f:
    label_names = json.load(f).get("label_names", [])

_cache = {}
def get_species_name(species_id):
    species_id = str(species_id)
    if species_id in _cache:
        return _cache[species_id]
    name = f"Species ID: {species_id}"
    try:
        r = requests.get(f"https://api.gbif.org/v1/species/{species_id}", timeout=10)
        if r.ok:
            d = r.json()
            name = d.get("canonicalName") or d.get("scientificName") or name
            family = d.get("family")
            if family:
                name += f" [{family}]"
        # common name (English), optional
        r2 = requests.get(f"https://api.gbif.org/v1/species/{species_id}/vernacularNames",
                          params={"limit": 50}, timeout=10)
        if r2.ok:
            for v in r2.json().get("results", []):
                if v.get("language") in ("eng", "en"):
                    name += f" — {v['vernacularName']}"
                    break
    except Exception as e:
        print("GBIF lookup failed:", e)
    _cache[species_id] = name
    return name

# 4. Predict
with torch.no_grad():
    probs = torch.softmax(model(input_tensor), dim=-1)[0]

top_probs, top_idx = torch.topk(probs, k=5)

print(f"\n🌿 Best match: {get_species_name(label_names[top_idx[0].item()])} "
      f"({top_probs[0].item()*100:.1f}%)\n")
print("Top 5:")
for rank, (p, i) in enumerate(zip(top_probs, top_idx), 1):
    print(f"{rank}. {get_species_name(label_names[i.item()])} — {p.item()*100:.2f}%")

Saving Mango.jpg to Mango (29).jpg

🌿 Best match: Sparbarus miccosukee [Caenidae] (10.4%)

Top 5:
1. Sparbarus miccosukee [Caenidae] — 10.39%
2. Smicra fulvomaculata [Chalcididae] — 5.82%
3. Eurymyiocnema [Pteromalidae] — 3.45%
4. Epilelaps [Pteromalidae] — 2.56%
5. Aphotismus niger [Pteromalidae] — 2.16%


In [38]:
import json, torch
from huggingface_hub import hf_hub_download

p = hf_hub_download(repo_id=model_id, filename="labels.json")
with open(p, encoding="utf-8") as f:
    labels = json.load(f)

# Sanity check: the class order should match config.json's label_names
assert len(labels) == len(label_names) == 7806
assert all(labels[i]["species_id"] == label_names[i] for i in range(len(labels))), "order mismatch"

# Look up by species_id so the mapping doesn't depend on list order
id_to_info = {d["species_id"]: d for d in labels}

def get_name(idx):
    d = id_to_info[label_names[idx]]
    return f"{d['species']} [{d['family']}]"

with torch.no_grad():
    probs = torch.softmax(model(input_tensor), dim=-1)[0]
top_probs, top_idx = torch.topk(probs, k=5)

print(f"\n🌿 Best match: {get_name(top_idx[0].item())} ({top_probs[0].item()*100:.1f}%)\n")
print("Top 5:")
for r, (p_, i) in enumerate(zip(top_probs, top_idx), 1):
    print(f"{r}. {get_name(i.item())} — {p_.item()*100:.2f}%")


🌿 Best match: Aloysia citrodora Paláu [Verbenaceae] (10.4%)

Top 5:
1. Aloysia citrodora Paláu [Verbenaceae] — 10.39%
2. Nerium oleander L. [Apocynaceae] — 5.82%
3. Ludwigia grandiflora (Michx.) Greuter & Burdet [Onagraceae] — 3.45%
4. Castanea crenata Siebold & Zucc. [Fagaceae] — 2.56%
5. Schinus terebinthifolia Raddi [Anacardiaceae] — 2.16%


In [35]:
from huggingface_hub import HfApi, hf_hub_download
model_id = "PlantNet/mobilenetv4-conv-s-plantclef2024"
print(HfApi().list_repo_files(model_id))
print(label_names[:10], len(label_names))

['.gitattributes', 'LICENSE', 'README.md', 'config.json', 'example.jpg', 'labels.json', 'model.safetensors', 'onnx/inference.py', 'onnx/metadata.json', 'onnx/model.onnx']
['1355868', '1355869', '1355870', '1355871', '1355872', '1355873', '1355880', '1355881', '1355882', '1355884'] 7806


In [36]:
import json
from huggingface_hub import hf_hub_download

p = hf_hub_download(repo_id=model_id, filename="labels.json")
with open(p, encoding="utf-8") as f:
    labels = json.load(f)

print(type(labels), len(labels))
if isinstance(labels, dict):
    print(list(labels.items())[:5])
else:
    print(labels[:5])


labels.json:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

<class 'list'> 7806
[{'species_id': '1355868', 'species': 'Lactuca virosa L.', 'genus': 'Lactuca', 'family': 'Asteraceae'}, {'species_id': '1355869', 'species': 'Crepis capillaris (L.) Wallr.', 'genus': 'Crepis', 'family': 'Asteraceae'}, {'species_id': '1355870', 'species': 'Crepis foetida L.', 'genus': 'Crepis', 'family': 'Asteraceae'}, {'species_id': '1355871', 'species': 'Hypochaeris glabra L.', 'genus': 'Hypochaeris', 'family': 'Asteraceae'}, {'species_id': '1355872', 'species': 'Hypochaeris radicata L.', 'genus': 'Hypochaeris', 'family': 'Asteraceae'}]


In [37]:
p = hf_hub_download(repo_id=model_id, filename="onnx/inference.py")
print(open(p).read())

inference.py:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

#!/usr/bin/env python3
"""ONNX Runtime inference example with no PyTorch or timm dependency."""

import json

import numpy as np
import onnxruntime as ort
from huggingface_hub import hf_hub_download
from PIL import Image


MODEL_ID = "PlantNet/mobilenetv4-conv-s-plantclef2024"


def softmax(values: np.ndarray) -> np.ndarray:
    values = values - values.max()
    probabilities = np.exp(values)
    return probabilities / probabilities.sum()


def main() -> None:
    model_path = hf_hub_download(MODEL_ID, "onnx/model.onnx")
    metadata_path = hf_hub_download(MODEL_ID, "onnx/metadata.json")
    labels_path = hf_hub_download(MODEL_ID, "labels.json")
    image_path = hf_hub_download(MODEL_ID, "example.jpg")

    with open(metadata_path, encoding="utf-8") as handle:
        metadata = json.load(handle)
    with open(labels_path, encoding="utf-8") as handle:
        labels = json.load(handle)

    preprocessing = metadata["preprocessing"]
    height, width = preprocessing["resize"]
    mean 

In [43]:
import requests
from google.colab import files

API_KEY = "2b10JS3pb0ei9In3LdiHEpLKu"
uploaded = files.upload()
img_path = next(iter(uploaded))

url = f"https://my-api.plantnet.org/v2/identify/all?api-key={"2b10JS3pb0ei9In3LdiHEpLKu"}&lang=en&nb-results=5"
with open(img_path, "rb") as f:
    r = requests.post(url, files=[("images", (img_path, f))], data={"organs": "leaf"})
r.raise_for_status()

for i, res in enumerate(r.json()["results"], 1):
    sp = res["species"]
    common = ", ".join(sp.get("commonNames", [])[:2])
    print(f"{i}. {sp['scientificNameWithoutAuthor']} [{sp['family']['scientificNameWithoutAuthor']}] "
          f"{common} — {res['score']*100:.1f}%")

Saving Mango.jpg to Mango (32).jpg
1. Mangifera indica [Anacardiaceae] Mango, Mango Tree — 95.8%
2. Sapium glandulosum [Euphorbiaceae] Gumtree, Milktree — 0.5%
3. Rauvolfia caffra [Apocynaceae] Quinine tree, Quininetree — 0.2%
4. Plumeria alba [Apocynaceae] White frangipani, Wild frangipani — 0.2%
5. Ficus insipida [Moraceae] Wild Fig, Sak kabah — 0.1%


In [41]:
import requests
from google.colab import files

API_KEY = "2b10JS3pb0ei9In3LdiHEpLKu"
uploaded = files.upload()
img_path = next(iter(uploaded))

url = f"https://my-api.plantnet.org/v2/identify/all?api-key={2b10JS3pb0ei9In3LdiHEpLKu}&lang=en&nb-results=5"
with open(img_path, "rb") as f:
    r = requests.post(url, files=[("images", (img_path, f))], data={"organs": "leaf"})
r.raise_for_status()

for i, res in enumerate(r.json()["results"], 1):
    sp = res["species"]
    common = ", ".join(sp.get("commonNames", [])[:2])
    print(f"{i}. {sp['scientificNameWithoutAuthor']} [{sp['family']['scientificNameWithoutAuthor']}] "
          f"{common} — {res['score']*100:.1f}%")

SyntaxError: invalid decimal literal (3812660497.py, line 8)

In [10]:
import requests, base64
from google.colab import files
import os

API_KEY = "Yu5taOkIfvB7tbQFPBmCYx9x7djvIfQG4pp7A6707d0y1IJ2D2"

# Check if Mango.jpg exists or prompt for upload
img_path = '/content/Mango.jpg'
if not os.path.exists(img_path):
    print("Please upload a plant/leaf image for health assessment:")
    uploaded = files.upload()
    img_path = next(iter(uploaded))

with open(img_path, "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode()

r = requests.post(
    "https://plant.id/api/v3/health_assessment",
    headers={"Api-Key": API_KEY, "Content-Type": "application/json"},
    json={
        "images": [img_b64],
        "similar_images": True
    },
)

if r.ok:
    print(r.json())
else:
    print(f"API request failed with status code {r.status_code}:")
    print(r.text)

API request failed with status code 401:
The specified api key not found.
